In [3]:
import pandas as pd
import numpy as np
import os
import re
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
full_data = pd.read_csv("full_animal_data.csv", low_memory = False)

In [5]:
dogs = full_data[full_data["animal_type_canonical"] == "Dog"].copy()

In [6]:
def clean_color(color):
    if pd.isna(color):
        return "Unknown"
    
    c = str(color).lower().strip()

    # Remove separators and punctuation
    for char in ["/", "-", ";", ","]:
        c = c.replace(char, " ")

    # Base colors
    if "black" in c or "blk" in c:
        return "Black"
    if "white" in c or "wht" in c:
        return "White"
    if "brown" in c or "chocolate" in c or "liver" in c:
        return "Brown"
    if "tan" in c:
        return "Tan"
    if "gray" in c or "grey" in c or "silver" in c:
        return "Gray"
    if "red" in c:
        return "Red"
    if "gold" in c or "yellow" in c or "buff" in c:
        return "Gold"
    if "cream" in c:
        return "Cream"
    if "blue" in c and "merle" not in c and "tick" not in c and "tiger" not in c:
        return "Blue"
    if "fawn" in c:
        return "Fawn"
    if "sable" in c:
        return "Sable"
    if "apricot" in c:
        return "Apricot"

    # Patterns
    if "brindle" in c:
        return "Brindle"
    if "merle" in c:
        return "Merle"
    if "tick" in c:
        return "Ticked"
    if "tri" in c:  # tricolor
        return "Tricolor"

    # Fallback
    return c.title()

dogs["primary_color_clean"] = dogs["primary_color"].apply(clean_color)
dogs["secondary_color_clean"] = dogs["secondary_color"].apply(clean_color)


In [7]:
# Count occurrences
color_counts = dogs["primary_color_clean"].value_counts()

# Keep colors that appear at least 200 times (others → "Other")
common_colors = color_counts[color_counts >= 200].index.tolist()

# Create grouped color column
dogs["primary_color_grouped"] = dogs["primary_color_clean"].apply(
    lambda x: x if x in common_colors else "Other"
)

dogs["primary_color_grouped"].value_counts()


primary_color_grouped
Black       38113
Brown       29089
White       26346
Tan         18236
Tricolor     5505
Red          5057
Gray         4355
Blue         4236
Gold         3135
Cream        2469
Brindle      1838
Fawn         1664
Sable        1506
Merle         931
Other         466
Ticked        368
Name: count, dtype: int64

<h3>Fixing Breed Issues</h3>

In [8]:
import pandas as pd
import numpy as np
import re

# 1. Extract first part of mix and strip "Mix"
def extract_primary(breed_str):
    if pd.isna(breed_str):
        return "Unknown"
    s = str(breed_str).strip()
    # drop trailing "Mix"
    s = re.sub(r'\s+Mix$', '', s, flags=re.IGNORECASE)
    # take first part before "/"
    s = s.split("/")[0].strip()
    return s

dogs["breed_primary_raw"] = dogs["breed"].apply(extract_primary)
dogs["breed_primary_raw"].value_counts().head(30)


breed_primary_raw
Unknown                  16628
Pit Bull                 14485
Labrador Retriever       13538
Chihuahua Shorthair      10118
German Shepherd           6992
CHIHUAHUA SH              5773
PIT BULL                  4520
Australian Cattle Dog     3153
GERM SHEPHERD             2855
Siberian Husky            2120
Dachshund                 2109
SIBERIAN HUSKY            1980
Boxer                     1885
Border Collie             1789
Great Pyrenees            1564
Miniature Poodle          1389
Australian Shepherd       1286
Catahoula                 1260
Beagle                    1221
LABRADOR RETR             1175
Yorkshire Terrier         1166
MALTESE                   1089
Miniature Schnauzer       1076
Jack Russell Terrier      1051
Rat Terrier               1044
Pointer                   1031
Staffordshire             1015
Rottweiler                 992
POODLE MIN                 988
Cairn Terrier              960
Name: count, dtype: int64

In [9]:
# normalize casing first
dogs["breed_primary_norm"] = dogs["breed_primary_raw"].str.strip()

# mapping for common abbreviations & variants
normalize_map = {
    "LABRADOR RETR": "Labrador Retriever",
    "LABRADOR RETRIE": "Labrador Retriever",
    "LABRADOR RETRIEVER": "Labrador Retriever",
    "LABRADOR RETRIE ": "Labrador Retriever",

    "CHIHUAHUA SH": "Chihuahua",
    "CHIHUAHUA SMOOT": "Chihuahua",
    "CHIHUAHUA LH": "Chihuahua Longhair",
    "CHIHUAHUA LONG": "Chihuahua Longhair",
    "CHIHUAHUA, OTHE": "Chihuahua",
    "CHIHUAHUA MIX;T": "Chihuahua",
    "CHIHUAHUA MIX;P": "Chihuahua",
    "CHIHUAHUA CHIHU": "Chihuahua",   # in case similar patterns exist

    "GERM SHEPHERD": "German Shepherd Dog",
    "GERM SHEPHERD ": "German Shepherd Dog",
    "GERM SHEPHERD DOG": "German Shepherd Dog",
    "GERM SHEPHERD DOG ": "German Shepherd Dog",
    "GERM SHEPHERD, AUSTR": "German Shepherd Dog",

    "GERM SH POINT": "German Shorthair Pointer",
    "GERM WH POINT": "German Wirehaired Pointer",

    "AUST CATTLE DOG": "Australian Cattle Dog",
    "AUST SHEPHERD": "Australian Shepherd",
    "AUST KELPIE": "Australian Kelpie",
    "AUST TERRIER": "Australian Terrier",

    "COCKER SPAN": "Cocker Spaniel",
    "ENG SPRNGR SPAN": "English Springer Spaniel",
    "ENG COCKER SPAN": "English Cocker Spaniel",
    "CAVALIER SPAN": "Cavalier King Charles Spaniel",

    "MIN PINSCHER": "Miniature Pinscher",
    "MIN PINSCH": "Miniature Pinscher",
    "SCHNAUZER MIN": "Miniature Schnauzer",
    "SCHNAUZER STAND": "Standard Schnauzer",
    "SCHNAUZER GIANT": "Schnauzer Giant",
    "POODLE MIN": "Miniature Poodle",
    "POODLE TOY": "Toy Poodle",
    "POODLE STND": "Standard Poodle",

    "YORKSHIRE TERR": "Yorkshire Terrier",
    "YORKSHIRE TERRI": "Yorkshire Terrier",

    "AM PIT BULL TER": "Pit Bull",
    "PIT BULL TERRIE": "Pit Bull",
    "PIT BULL": "Pit Bull",
    "PIT BULL ": "Pit Bull",

    "AUST KELPIE": "Australian Kelpie",
    "GERM SHEPHERD DOG": "German Shepherd Dog",

    "ST BERNARD SMTH": "St. Bernard Smooth Coat",
    "ST BERNARD RGH": "St. Bernard Rough Coat",
    "GREAT PYRENEES": "Great Pyrenees",
    "GREAT PYRENEESE": "Great Pyrenees",

    "BICHON FRISE": "Bichon Frise",
    "SC WHEAT TERR": "Soft Coated Wheaten Terrier",

    "BOXER, MIX": "Boxer",
    "POMERANIAN -MIX": "Pomeranian",
}

# apply mapping on UPPERCASE versions
dogs["breed_primary_norm"] = (
    dogs["breed_primary_norm"]
    .apply(lambda s: normalize_map.get(s.upper(), s))
)
dogs["breed_primary_norm"].value_counts().head(30)


breed_primary_norm
Pit Bull                 19115
Unknown                  16628
Labrador Retriever       14714
Chihuahua Shorthair      10118
German Shepherd           6992
Chihuahua                 5779
Australian Cattle Dog     3546
German Shepherd Dog       2855
Miniature Poodle          2377
Siberian Husky            2120
Dachshund                 2109
SIBERIAN HUSKY            1980
Boxer                     1885
Border Collie             1789
Chihuahua Longhair        1752
Yorkshire Terrier         1746
Great Pyrenees            1666
Australian Shepherd       1509
Catahoula                 1260
Miniature Schnauzer       1245
Beagle                    1221
MALTESE                   1089
Jack Russell Terrier      1051
Rat Terrier               1044
Pointer                   1031
Staffordshire             1015
Rottweiler                 992
Cairn Terrier              960
Anatol Shepherd            888
Shih Tzu                   872
Name: count, dtype: int64

In [10]:
normalize_map.update({
    "SIBERIAN HUSKY": "Siberian Husky",
    "MALTESE": "Maltese",
    "GERMAN SHEPHERD DOG": "German Shepherd",
    "SHIH TZU": "Shih Tzu",
    "FRENCH BULLDOG" : "French Bulldog",
    "BORDER COLLIE" : "Border Collie",
    "POMERANIAN" : "Pomeranian",
    "DACHSHUND" : "Dachshund",
    "PARSON RUSS TER" : "Parson Russell Terrier",
    "CAIRN TERRIER" : "Cairn Terrier",
    "PUG" : "Pug",
    "DOBERMAN PINSCH" : "Doberman Pinscher",
    "NORFOLK TERRIER" : "Norfolk Terrier",
    "BORDER TERRIER" : "Border Terrier",
    "BELG MALINOIS" : "Belgium Malinois",
})

dogs["breed_primary_norm"] = (
    dogs["breed_primary_norm"]
    .apply(lambda x: normalize_map.get(str(x).upper(), x))
)

dogs["breed_primary_norm"].value_counts().head(60)


breed_primary_norm
Pit Bull                          19115
Unknown                           16628
Labrador Retriever                14714
Chihuahua Shorthair               10118
German Shepherd                    9847
Chihuahua                          5779
Siberian Husky                     4100
Australian Cattle Dog              3546
Dachshund                          2555
Miniature Poodle                   2377
Border Collie                      2250
Boxer                              1885
Chihuahua Longhair                 1752
Yorkshire Terrier                  1746
Maltese                            1679
Great Pyrenees                     1666
Australian Shepherd                1509
Shih Tzu                           1380
Cairn Terrier                      1331
Catahoula                          1260
Miniature Schnauzer                1245
Beagle                             1221
Jack Russell Terrier               1051
Rat Terrier                        1044
Pointer              

In [11]:
breed_corrections = {
    # ---- Pit bull & bully-type normalization (Option A = all -> "Pit Bull") ----
    "Pit Bull": "Pit Bull",
    "PIT BULL": "Pit Bull",
    "Pitbull": "Pit Bull",
    "Pit Bull Terrier": "Pit Bull",
    "PIT BULL TERRIE": "Pit Bull",
    "AM PIT BULL TER": "Pit Bull",
    "AMERICAN PIT BU": "Pit Bull",
    "American Pit Bull Terrier": "Pit Bull",
    "American Pit Bull": "Pit Bull",

    # sometimes shelter codes where primary is clearly Pit Bull
    "AM PIT BULL TERRIER": "Pit Bull",
    "AMERICAN PIT BULL TERRIER": "Pit Bull",

    # ---- Chihuahua variants ----
    "CHIHUAHUA SH": "Chihuahua Shorthair",
    "Chihuahua Sh": "Chihuahua Shorthair",
    "CHIHUAHUA SMOOT": "Chihuahua Shorthair",
    "CHIHUAHUA LONG": "Chihuahua Longhair",
    "Chihuahua LH": "Chihuahua Longhair",
    "CHIHUAHUA LH": "Chihuahua Longhair",
    "CHIHUAHUA, OTHE": "Chihuahua",
    "CHIHUAHUA MIX;T": "Chihuahua",
    "CHIHUAHUA MIX;P": "Chihuahua",
    "DACHSHUND CHIHU": "Dachshund",  # take first breed
    "Chihuahua Shorthair": "Chihuahua Shorthair",
    "Chihuahua Longhair": "Chihuahua Longhair",

    # ---- Lab / shepherd / common big breeds ----
    "LABRADOR RETR": "Labrador Retriever",
    "LABRADOR RETRIE": "Labrador Retriever",
    "Labrador Retr": "Labrador Retriever",
    "Labrador Retr.": "Labrador Retriever",

    "GERM SHEPHERD": "German Shepherd",
    "SHEPHERD DOG, G": "German Shepherd Dog",
    "SHEPHERD, AUSTR": "Australian Shepherd",
    "AUST SHEPHERD": "Australian Shepherd",
    "AUST CATTLE DOG": "Australian Cattle Dog",
    "AUST KELPIE": "Australian Kelpie",
    "AUST TERRIER": "Australian Terrier",

    'Golden Retr': 'Golden Retriever',
    'GOLDEN RETR': 'Golden Retriever',

    # ---- Dachshund variants ----
    "DACHSHUND LH": "Dachshund Longhair",
    "DACHSHUND WH": "Dachshund Wirehair",
    "Dachshund Stan": "Dachshund",
    "DACHSHUND STAN": "Dachshund",
    "DACHSHUND CHIHU": "Dachshund",  # first breed
    
    # ---- Poodle variants ----
    "POODLE MIN": "Miniature Poodle",
    "POODLE TOY": "Toy Poodle",
    "POODLE STND": "Standard Poodle",
    "Poodle Min": "Miniature Poodle",
    "Poodle Toy": "Toy Poodle",
    "Poodle Stnd": "Standard Poodle",

    # ---- Terrier & small-dog codes ----
    "PARSON RUSS TER": "Parson Russell Terrier",
    "JACK RUSS TERR": "Jack Russell Terrier",
    "YORKSHIRE TERR": "Yorkshire Terrier",
    "YORKSHIRE TERRI": "Yorkshire Terrier",
    "SCOT TERRIER": "Scottish Terrier",
    "BEDLINGTON TERR": "Bedlington Terrier",
    "LAKELAND TERR": "Lakeland Terrier",
    "SEALYHAM TERR": "Sealyham Terr",
    "PATERDALE TERR": "Patterdale Terr",
    "PATTERDALE TERR": "Patterdale Terr",
    "TERRIER": "Terrier",
    "MIX;TERRIER": "Terrier",
    "MIX;TERRIER MIX": "Terrier",

    # ---- Misc common purebreds (uppercase codes -> nice case) ----
    "BEAGLE": "Beagle",
    "BEAGLE, MIX": "Beagle",
    "BEAGLE MIX": "Beagle",
    "SHIBA INU": "Shiba Inu",
    "BULL TERRIER": "Bull Terrier",
    "BOXER": "Boxer",
    "BOXER, MIX": "Boxer",
    "RAT TERRIER": "Rat Terrier",
    "PAPILLON": "Papillon",
    "AMER BULLDOG": "American Bulldog",
    "DOBERMAN PINSCH": "Doberman Pinscher",
    "YORKSHIRE TERRIER": "Yorkshire Terrier",
    "YORKSHIRE TERRI": "Yorkshire Terrier",
    "SIBERIAN HUSKY": "Siberian Husky",
    "FOX TERR WIRE": "Wire Hair Fox Terrier",
    "SILKY TERRIER": "Silky Terrier",
    "LHASA APSO": "Lhasa Apso",
    "STAFFORDSHIRE": "Staffordshire",
    "BASSET HOUND": "Basset Hound",
    "PEKINGESE": "Pekingese",
    "ROTTWEILER": "Rottweiler",
    "HAVANESE": "Havanese",
    "BICHON FRISE": "Bichon Frise",
    "CHINESE SHARPEI": "Chinese Sharpei",
    "NORFOLK TERRIER": "Norfolk Terrier",
    "CAVALIER SPAN": "Cavalier King Charles Spaniel",
    "CAVALIER SPANIEL": "Cavalier King Charles Spaniel",
    "CAVALIER KING CHARLES SPAN": "Cavalier King Charles Spaniel",
    "ALASKAN HUSKY": "Alaskan Husky",
    "ALASK MALAMUTE": "Alaskan Malamute",
    "ENG BULLDOG": "English Bulldog",
    "BOSTON TERRIER": "Boston Terrier",
    "SCHNAUZER MIN": "Miniature Schnauzer",
    "SCHNAUZER STAND": "Standard Schnauzer",
    "SCHNAUZER GIANT": "Schnauzer Giant",
    "ENG COCKER SPAN": "English Cocker Spaniel",
    "ENG SPRNGR SPAN": "English Springer Spaniel",
    "ENG POINTER": "English Pointer",
    "ENG FOXHOUND": "English Foxhound",
    "ENG COONHOUND": "English Coonhound",
    "ENG SHEPHERD": "English Shepherd",

    # ---- “AUST …” and similar abbreviations ----
    "AUST KELPIE": "Australian Kelpie",
    "AUST TERRIER": "Australian Terrier",
    "QUEENSLAND HEEL": "Queensland Heeler",
    "WELSH CORGI PEM": "Pembroke Welsh Corgi",
    "WELSH CORGI CAR": "Cardigan Welsh Corgi",
    "CARDIGAN  WELSH": "Cardigan Welsh Corgi",

    # ---- Hound & sighthound stuff ----
    "REDBONE HOUND": "Redbone Hound",
    "REDBONE HOUND/": "Redbone Hound",
    "PLOTT HOUND": "Plott Hound",
    "PBGV": "Pbgv",
    "TR WALKER HOUND": "Treeing Walker Coonhound",
    "TENN TR BRINDLE": "Treeing Tennesse Brindle",
    "BLUETICK HOUND": "Bluetick Hound",
    "ENG COONHOUND": "English Coonhound",

    # ---- Working & guardian breeds ----
    "NEAPOLITAN MAST": "Neapolitan Mastiff",
    "SPANISH MASTIFF": "Spanish Mastiff",
    "MASTIFF": "Mastiff",
    "BULLMASTIFF": "Bullmastiff",
    "DOGUE DE BORDX": "Dogue De Bordeaux",
    "DOGUE DE BORDX ": "Dogue De Bordeaux",
    "DOGUE DE BORDX": "Dogue De Bordeaux",
    "AKBASH": "Akbash",
    "KUVASZ": "Kuvasz",
    "KANGAL": "Kangal",
    "GR SWISS MTN": "Greater Swiss Mountain Dog",
    "GR SWISS MTN DOG": "Greater Swiss Mountain Dog",
    "ST BERNARD SMTH": "St. Bernard Smooth Coat",
    "ST BERNARD RGH": "St. Bernard Rough Coat",
    "BERNESE MTN DOG": "Bernese Mountain Dog",
    "NEWFOUNDLAND": "Newfoundland",
    "LANDSEER": "Landseer",
    "OLD ENG BULLDOG": "Old English Bulldog",
    "OLDENG SHEEPDOG": "Old English Sheepdog",
    "AKITA INU": "Akita",

    # ---- Misc less-common breeds & codes ----
    "PORT WATER DOG": "Port Water Dog",
    "CHESA BAY RETR": "Chesa Bay Retr",
    "Chesa Bay Retr": "Chesa Bay Retr",
    "NS DUCK TOLLING": "Nova Scotia Duck Tolling Retriever",
    "NS DUCK TOLLING RETR": "Nova Scotia Duck Tolling Retriever",
    "GORDON SETTER": "Gordon Setter",
    "BELG TERVUREN": "Belgian Tervuren",
    "Belgium Malinois": "Belgian Malinois",
    "BELG MALINOIS": "Belgian Malinois",
    "BELG SHEEPDOG": "Belgian Sheepdog",
    "PODENGO PEQUENO": "Podengo Pequeno",
    "KARELIAN BEAR": "Karelian Bear Dog",
    "SC WHEAT TERR": "Soft Coated Wheaten Terrier",
    "GR SWISS MTN": "Greater Swiss Mountain Dog",
    "SWED VALLHUND": "Swedish Vallhund",
    "NORW BUHUND": "Norwegian Buhund",
    "NS DUCK TOLLING": "Nova Scotia Duck Tolling Retriever",
    "COYOTE HYBRID": "Wolf Hybrid",
    "WELSH SPR SPAN": "Welsh Springer Spaniel",
    "SUSSEX SPAN": "Sussex Spaniel",
    "SPANISH MASTIFF": "Spanish Mastiff",
    "PUMI": "Pumi",
    "PLOTT HOUND": "Plott Hound",
    "TENN TR BRINDLE": "Treeing Tennesse Brindle",
    "POMERANIAN -MIX": "Pomeranian",
    "SHIH TZU -MIXED": "Shih Tzu",
    "AMER FOXHOUND": "American Foxhound",
    "AMER ESKIMO": "American Eskimo",
    "AMERICAN STAFF": "American Staffordshire Terrier",
    "AMERICAN COCKER": "American Cocker Spaniel",
    "AMERICAN PIT BU": "Pit Bull",

    # ---- Hairless, spitz, and other rarer codes ----
    "MEX HAIRLESS": "Mexican Hairless",
    "KOREAN JINDO": "Jindo",
    "JINDO": "Jindo",
    "CHINESE CRESTED": "Chinese Crested",
    "AKITA INU": "Akita",
    "SAMOYED": "Samoyed",
    "HOVAWART": "Hovawart",
    "PULI": "Puli",
    "PBGV": "Pbgv",

    # ---- Random text codes / placeholders / mix flags -> Unknown ----
    "Unknown": "Unknown",
    "UNKNOWN": "Unknown",
    "NaN": "Unknown",
    "0": "Unknown",
    "MIX": "Unknown",
    "MIXED BREED;TER": "Unknown",
    "MIXED BREED": "Unknown",
    "CHIHUAHUA MIX;P": "Unknown",
    "CHIHUAHUA MIX;T": "Unknown",
    "MALTI POO": "Unknown",  # you could map to Maltese/Poodle, but for primary "Unknown" is safest
    "COCKAPOO": "Unknown",
}
dogs["breed_primary_clean"] = (
    dogs["breed_primary_raw"]
      .str.strip()
      .map(breed_corrections)      # apply corrections
      .fillna(dogs["breed_primary_raw"])  # keep original if not in dict
)
dogs["breed_primary_norm"].unique()

array(['Black Mouth Cur', 'Chihuahua Shorthair', 'Shiba Inu',
       'Cocker Spaniel', 'American Pit Bull Terrier', 'Golden Retriever',
       'Miniature Pinscher', 'Pit Bull', 'Labrador Retriever', 'Havanese',
       'Rat Terrier', 'Pekingese', 'Basset Hound', 'Shih Tzu',
       'Lhasa Apso', 'Australian Cattle Dog', 'Miniature Schnauzer',
       'Dachshund', 'Pembroke Welsh Corgi', 'Carolina Dog', 'Rottweiler',
       'Border Collie', 'Cairn Terrier', 'Great Pyrenees',
       'Miniature Poodle', 'Redbone Hound', 'Pointer', 'Chinese Sharpei',
       'German Shepherd', 'Boxer', 'Chihuahua Longhair',
       'Australian Kelpie', 'Beagle', 'Harrier', 'Bruss Griffon',
       'Siberian Husky', 'English Pointer', 'Toy Poodle',
       'Australian Shepherd', 'Bull Terrier', 'Schnauzer Giant',
       'Yorkshire Terrier', 'Queensland Heeler', 'Pug', 'Plott Hound',
       'Jack Russell Terrier', 'Black', 'Cardigan Welsh Corgi',
       'Doberman Pinscher', 'Rhod Ridgeback', 'Jindo', 'Bichon Frise'

In [12]:
import pandas as pd
import numpy as np

# Map primary breeds to AKC-ish groups + Pit Bull group
akc_group_map = {
    # Pit bull–type
    "Pit Bull": "Terrier/Pit Bull",
    "American Pit Bull Terrier": "Terrier/Pit Bull",
    "American Staffordshire Terrier": "Terrier/Pit Bull",
    "Staffordshire": "Terrier/Pit Bull",

    # Sporting
    "Labrador Retriever": "Sporting",
    "Golden Retriever": "Sporting",
    "Cocker Spaniel": "Sporting",
    "English Springer Spaniel": "Sporting",
    "Chesa Bay Retr": "Sporting",
    "Weimaraner": "Sporting",
    "German Shorthair Pointer": "Sporting",
    "German Wirehaired Pointer": "Sporting",
    "English Setter": "Sporting",
    "Gordon Setter": "Sporting",
    "Brittany": "Sporting",
    "Pointer": "Sporting",
    "Vizsla": "Sporting",

    # Hound
    "Beagle": "Hound",
    "Basset Hound": "Hound",
    "Plott Hound": "Hound",
    "Bluetick Hound": "Hound",
    "Redbone Hound": "Hound",
    "Treeing Walker Coonhound": "Hound",
    "Treeing Tennesse Brindle": "Hound",
    "Greyhound": "Hound",
    "Whippet": "Hound",
    "Dachshund": "Hound",
    "Dachshund Longhair": "Hound",
    "Dachshund Wirehair": "Hound",
    "Harrier": "Hound",
    "Ibizan Hound": "Hound",
    "Pharaoh Hound": "Hound",
    "Saluki": "Hound",
    "BorzoI": "Hound",
    "Irish Wolfhound": "Hound",
    "Bloodhound": "Hound",

    # Working
    "Boxer": "Working",
    "Rottweiler": "Working",
    "Great Pyrenees": "Working",
    "Bernese Mountain Dog": "Working",
    "Newfoundland": "Working",
    "Great Dane": "Working",
    "Mastiff": "Working",
    "Bullmastiff": "Working",
    "Neapolitan Mastiff": "Working",
    "Doberman Pinscher": "Working",
    "Siberian Husky": "Working",
    "Alaskan Malamute": "Working",
    "Akita": "Working",
    "Anatol Shepherd": "Working",           # livestock guardian → treat as Working
    "Boerboel": "Working",
    "Cane Corso": "Working",
    "Samoyed": "Working",
    "St. Bernard Rough Coat": "Working",
    "St. Bernard Smooth Coat": "Working",
    "Greater Swiss Mountain Dog": "Working",

    # Herding
    "German Shepherd": "Herding",
    "German Shepherd Dog": "Herding",
    "Australian Shepherd": "Herding",
    "Australian Cattle Dog": "Herding",
    "Border Collie": "Herding",
    "Cardigan Welsh Corgi": "Herding",
    "Pembroke Welsh Corgi": "Herding",
    "Shetland Sheepdog": "Herding",
    "Australian Kelpie": "Herding",
    "Catahoula": "Herding",  # technically not AKC herding, but “herding-type” for analysis
    "Belgian Malinois": "Herding",
    "Belgian Tervuren": "Herding",
    "Belgian Sheepdog": "Herding",
    "Collie Rough": "Herding",
    "Collie Smooth": "Herding",

    # Terrier
    "Jack Russell Terrier": "Terrier",
    "Parson Russell Terrier": "Terrier",
    "Rat Terrier": "Terrier",
    "Cairn Terrier": "Terrier",
    "Norwich Terrier": "Terrier",
    "West Highland": "Terrier",
    "Scottish Terrier": "Terrier",
    "Miniature Schnauzer": "Terrier",
    "Schnauzer Giant": "Terrier",
    "Airedale Terrier": "Terrier",
    "Bull Terrier": "Terrier",
    "Bull Terrier Miniature": "Terrier",
    "Border Terrier": "Terrier",
    "Staffordshire Bull Terrier": "Terrier",  # if present
    "Soft Coated Wheaten Terrier": "Terrier",
    "Bedlington Terr": "Terrier",
    "Lakeland Terrier": "Terrier",
    "Sealyham Terr": "Terrier",
    "Irish Terrier": "Terrier",

    # Toy
    "Chihuahua": "Toy",
    "Chihuahua Shorthair": "Toy",
    "Chihuahua Longhair": "Toy",
    "Maltese": "Toy",
    "Yorkshire Terrier": "Toy",
    "Pomeranian": "Toy",
    "Papillon": "Toy",
    "Pekingese": "Toy",
    "Havanese": "Toy",
    "Miniature Pinscher": "Toy",
    "Toy Poodle": "Toy",
    "Cavalier King Charles Spaniel": "Toy",
    "Toy Fox Terrier": "Toy",
    "Shih Tzu": "Toy",
    "Bruss Griffon": "Toy",
    "Affenpinscher": "Toy",
    "Silky Terrier": "Toy",
    "Japanese Chin": "Toy",

    # Non-Sporting
    "French Bulldog": "Non-Sporting",
    "Miniature Poodle": "Non-Sporting",
    "Standard Poodle": "Non-Sporting",
    "Boston Terrier": "Non-Sporting",
    "Bulldog": "Non-Sporting",
    "Lhasa Apso": "Non-Sporting",
    "Chinese Sharpei": "Non-Sporting",
    "Dalmatian": "Non-Sporting",
    "Chow Chow": "Non-Sporting",
    "Keeshond": "Non-Sporting",
    "Shiba Inu": "Non-Sporting",
    "Coton De Tulear": "Non-Sporting",
    "Schipperke": "Non-Sporting",
    "Tibetan Terrier": "Non-Sporting",
    "Tibetan Spaniel": "Non-Sporting",
    "Bichon Frise": "Non-Sporting",

    # Foundation Stock / misc: you can shove into closest AKC-type
    "Black Mouth Cur": "Foundation/Working",
    "Carolina Dog": "Foundation/Working",
    "Blue Lacy": "Foundation/Working",
    "American Bulldog": "Foundation/Working",
    "Canaan Dog": "Foundation/Working",
    "Feist": "Foundation/Working",
    "Treeing Cur": "Foundation/Working",
}

# Create the group variable
dogs["akc_group"] = (
    dogs["breed_primary_clean"]
      .map(akc_group_map)
      .fillna("Other/Unknown")
)

# Optional: make it a categorical with a nice order
akc_order = [
    "Sporting", "Hound", "Working", "Herding",
    "Terrier", "Terrier/Pit Bull", "Toy",
    "Non-Sporting", "Foundation/Working", "Other/Unknown"
]
dogs["akc_group"] = pd.Categorical(dogs["akc_group"], categories=akc_order, ordered=False)


In [13]:
dogs["breed_akc_group"] = dogs["breed_primary_norm"].map(akc_group_map)
dogs["breed_akc_group"] = dogs["breed_akc_group"].fillna("Other/Unknown")


In [14]:
dogs["breed_akc_group"].value_counts()


breed_akc_group
Other/Unknown         27548
Toy                   24776
Terrier/Pit Bull      21310
Herding               20266
Sporting              17439
Working               11599
Terrier                6593
Hound                  6239
Non-Sporting           5299
Foundation/Working     2245
Name: count, dtype: int64

In [15]:
# Count breeds except Unknown
breed_counts = (
    dogs.loc[dogs["breed_primary_clean"] != "Unknown", "breed_primary_clean"]
        .value_counts()
)

# Actual top-20 breeds
top20_breeds = breed_counts.head(20).index.tolist()
print(top20_breeds)

# Create new variable
dogs["breed_top20"] = np.where(
    dogs["breed_primary_clean"].isin(top20_breeds),
    dogs["breed_primary_clean"],
    "Other breeds"
)

# Drop unknown
dogs = dogs[dogs["breed_primary_norm"] != "Unknown"]

# Identify top 20
top20 = (
    dogs["breed_primary_norm"]
    .value_counts()
    .head(20)
    .index
)

# Create top 20 + Other
dogs["breed_top20"] = dogs["breed_primary_norm"].apply(
    lambda b: b if b in top20 else "Other"
)

# Confirm counts
dogs["breed_top20"].value_counts()



['Pit Bull', 'Chihuahua Shorthair', 'Labrador Retriever', 'German Shepherd', 'Siberian Husky', 'Australian Cattle Dog', 'Miniature Poodle', 'Boxer', 'Dachshund', 'Border Collie', 'Chihuahua Longhair', 'Yorkshire Terrier', 'Great Pyrenees', 'Australian Shepherd', 'Beagle', 'Rat Terrier', 'Catahoula', 'Miniature Schnauzer', 'Rottweiler', 'Staffordshire']


breed_top20
Other                    36832
Pit Bull                 19115
Labrador Retriever       14714
Chihuahua Shorthair      10118
German Shepherd           9847
Chihuahua                 5779
Siberian Husky            4100
Australian Cattle Dog     3546
Dachshund                 2555
Miniature Poodle          2377
Border Collie             2250
Boxer                     1885
Chihuahua Longhair        1752
Yorkshire Terrier         1746
Maltese                   1679
Great Pyrenees            1666
Australian Shepherd       1509
Shih Tzu                  1380
Cairn Terrier             1331
Catahoula                 1260
Miniature Schnauzer       1245
Name: count, dtype: int64

<h3>Chi Square Test</h3>

In [9]:
from scipy.stats import chi2_contingency

def chi_square_test(df, column):
    contingency = pd.crosstab(df[column], df["outcome_canonical"])
    chi2, p, dof, exp = chi2_contingency(contingency)
    return p

# Example:
print("Sex:", chi_square_test(dogs, "sex"))
print("Age Group:", chi_square_test(dogs, "age_group"))
print("Shelter:", chi_square_test(dogs, "shelter"))
print("Primary Color:", chi_square_test(dogs, "primary_color_grouped"))


Sex: 0.0
Age Group: 0.0
Shelter: 0.0
Primary Color: 0.0


<h3> Sex Model </h3>

In [17]:
m_sex = smf.logit("is_adopted ~ C(sex)", data=dogs).fit()
print(m_sex.summary())
print(np.exp(m_sex.params))  # Odds ratios


Optimization terminated successfully.
         Current function value: 0.527717
         Iterations 8
                           Logit Regression Results                           
Dep. Variable:             is_adopted   No. Observations:               143313
Model:                          Logit   Df Residuals:                   143306
Method:                           MLE   Df Model:                            6
Date:                Sun, 30 Nov 2025   Pseudo R-squ.:                  0.2269
Time:                        21:00:17   Log-Likelihood:                -75629.
converged:                       True   LL-Null:                       -97824.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                  -3.1778      0.055    -57.412      0.000      -3.286   

<h3> Sterlized Model </h3>

In [19]:
dogs["fixed"] = dogs["sex"].str.contains("Spayed|Neutered", na=False).astype(int)

m_fixed = smf.logit("is_adopted ~ fixed", data=dogs).fit()
print(m_fixed.summary())
print(np.exp(m_fixed.params))


Optimization terminated successfully.
         Current function value: 0.529511
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:             is_adopted   No. Observations:               143314
Model:                          Logit   Df Residuals:                   143312
Method:                           MLE   Df Model:                            1
Date:                Sun, 30 Nov 2025   Pseudo R-squ.:                  0.2243
Time:                        21:00:40   Log-Likelihood:                -75886.
converged:                       True   LL-Null:                       -97825.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -2.9693      0.022   -133.619      0.000      -3.013      -2.926
fixed          3.3458      0.

<h3> Age Model </h3>

In [23]:
m_age = smf.logit("is_adopted ~ age_years", data=dogs).fit()
print(np.exp(m_age.params))
print(m_age.summary())

Optimization terminated successfully.
         Current function value: 0.600916
         Iterations 6
Intercept    0.79984
age_years    0.85684
dtype: float64
                           Logit Regression Results                           
Dep. Variable:             is_adopted   No. Observations:                43252
Model:                          Logit   Df Residuals:                    43250
Method:                           MLE   Df Model:                            1
Date:                Sun, 30 Nov 2025   Pseudo R-squ.:                 0.04439
Time:                        21:01:54   Log-Likelihood:                -25991.
converged:                       True   LL-Null:                       -27198.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.2233      0.015    -15.196      0.

<h3> Age Group Model </h3>

In [28]:
dogs_clean = dogs[dogs["age_years"] < 20].copy()

In [29]:
m_agegroup = smf.logit("is_adopted ~ C(age_group)", data=dogs_clean).fit()
print(np.exp(m_agegroup.params))
print(m_agegroup.summary())

Optimization terminated successfully.
         Current function value: 0.601020
         Iterations 7
Intercept                 0.030817
C(age_group)[T.1–<3]     20.901733
C(age_group)[T.3–<7]     13.088514
C(age_group)[T.7–<15]     5.481486
C(age_group)[T.<1]       23.342069
dtype: float64
                           Logit Regression Results                           
Dep. Variable:             is_adopted   No. Observations:                43230
Model:                          Logit   Df Residuals:                    43225
Method:                           MLE   Df Model:                            4
Date:                Sun, 30 Nov 2025   Pseudo R-squ.:                 0.04441
Time:                        21:08:05   Log-Likelihood:                -25982.
converged:                       True   LL-Null:                       -27190.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                            coef    std err          z      P>|z|      [0.02

<h3>Color Model </h3>

In [26]:
m_color = smf.logit("is_adopted ~ C(primary_color_grouped)", data=dogs).fit()
print(np.exp(m_color.params))
print(m_color.summary())

Optimization terminated successfully.
         Current function value: 0.679938
         Iterations 5
Intercept                               0.800671
C(primary_color_grouped)[T.Blue]        1.127760
C(primary_color_grouped)[T.Brindle]     0.656089
C(primary_color_grouped)[T.Brown]       0.972842
C(primary_color_grouped)[T.Cream]       0.864594
C(primary_color_grouped)[T.Fawn]        0.837651
C(primary_color_grouped)[T.Gold]        0.849368
C(primary_color_grouped)[T.Gray]        0.559890
C(primary_color_grouped)[T.Merle]       1.153475
C(primary_color_grouped)[T.Other]       0.472976
C(primary_color_grouped)[T.Red]         1.070531
C(primary_color_grouped)[T.Sable]       1.282572
C(primary_color_grouped)[T.Tan]         0.922456
C(primary_color_grouped)[T.Ticked]      2.549515
C(primary_color_grouped)[T.Tricolor]    1.068366
C(primary_color_grouped)[T.White]       0.819368
dtype: float64
                           Logit Regression Results                           
Dep. Variable:      

<h3>Breed Regression: AKC Groups</h3>

In [102]:
import statsmodels.formula.api as smf

model_akc = smf.logit(
    formula="is_adopted ~ C(breed_akc_group)",
    data=dogs
).fit()

print(model_akc.summary())


Optimization terminated successfully.
         Current function value: 0.667441
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:             is_adopted   No. Observations:               143314
Model:                          Logit   Df Residuals:                   143304
Method:                           MLE   Df Model:                            9
Date:                Sun, 30 Nov 2025   Pseudo R-squ.:                 0.02219
Time:                        22:56:53   Log-Likelihood:                -95654.
converged:                       True   LL-Null:                       -97825.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                             coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------------------
Intercept                                  0.4092   

<h3> Full Regression </h3>

In [105]:
import pandas as pd
import statsmodels.formula.api as smf

dogs_clean = dogs[dogs["age_years"] < 20].copy()

# 1. Make sure is_adopted exists
dogs_clean["is_adopted"] = (dogs_clean["outcome_canonical"] == "Adoption").astype(int)

# 2. (Optional but recommended) drop Unknown from primary breed
dogs_clean = dogs_clean[dogs_clean["breed_primary_norm"] != "Unknown"].copy()

# 3. Build Top 20 breed variable
top20 = (
    dogs_clean["breed_primary_norm"]
    .value_counts()
    .head(20)
    .index
)

dogs_clean["breed_top20"] = dogs_clean["breed_primary_norm"].apply(
    lambda b: b if b in top20 else "Other"
)

# (Optional) make it a categorical with a clear reference, e.g. "Other"
dogs_clean["breed_top20"] = pd.Categorical(
    dogs_clean["breed_top20"],
    categories=sorted(list(top20) + ["Other"]),  # or put "Other" first if you prefer it as ref
    ordered=False
)

# 4. Choose predictors you want in the model (now including breed_top20)
model_data = dogs_clean[[
    "is_adopted",
    "sex",
    "age_group",
    "primary_color_grouped",
    "shelter",
    "age_years",        # numeric
    "breed_top20"       # NEW
]].dropna()

# 5. Fit full logistic regression with breed
logit_model = smf.logit(
    formula="""
        is_adopted ~ 
        C(sex) + 
        C(age_group) + 
        C(primary_color_grouped) + 
        C(shelter) +
        C(breed_top20) +
        age_years
    """,
    data=model_data
).fit()

print(logit_model.summary())

         Current function value: 0.464007
         Iterations: 35


/opt/conda/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


                           Logit Regression Results                           
Dep. Variable:             is_adopted   No. Observations:                26938
Model:                          Logit   Df Residuals:                    26890
Method:                           MLE   Df Model:                           47
Date:                Sun, 30 Nov 2025   Pseudo R-squ.:                  0.2830
Time:                        23:05:41   Log-Likelihood:                -12499.
converged:                      False   LL-Null:                       -17433.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                               coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------------
Intercept                                   -6.5393      0.492    -13.298      0.000      -7.503      -5.575
C(sex)[T.Intact Female]                  

In [106]:
import numpy as np

odds_ratios = np.exp(logit_model.params)
print(odds_ratios)


Intercept                                   1.445530e-03
C(sex)[T.Intact Female]                     3.649962e+00
C(sex)[T.Intact Male]                       6.832867e+00
C(sex)[T.Male]                              1.606085e+00
C(sex)[T.Neutered Male]                     2.248842e+02
C(sex)[T.Spayed Female]                     2.426557e+02
C(sex)[T.Unknown]                           4.058185e-08
C(age_group)[T.1–<3]                        2.279632e+00
C(age_group)[T.3–<7]                        2.223690e+00
C(age_group)[T.7–<15]                       1.916182e+00
C(age_group)[T.<1]                          2.767616e+00
C(primary_color_grouped)[T.Blue]            1.257934e+00
C(primary_color_grouped)[T.Brindle]         1.156315e+00
C(primary_color_grouped)[T.Brown]           1.063134e+00
C(primary_color_grouped)[T.Cream]           1.276729e+00
C(primary_color_grouped)[T.Fawn]            9.975667e-01
C(primary_color_grouped)[T.Gold]            9.440707e-01
C(primary_color_grouped)[T.Gray

<h3>Interaction Models</h3>

In [32]:
m_age_shelter = smf.logit(
    "is_adopted ~ age_years * C(shelter)",
    data=dogs
).fit()

print(m_age_shelter.summary())
print("\nOdds Ratios:\n", np.exp(m_age_shelter.params))


Optimization terminated successfully.
         Current function value: 0.594617
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:             is_adopted   No. Observations:                43252
Model:                          Logit   Df Residuals:                    43246
Method:                           MLE   Df Model:                            5
Date:                Sun, 30 Nov 2025   Pseudo R-squ.:                 0.05441
Time:                        21:16:55   Log-Likelihood:                -25718.
converged:                       True   LL-Null:                       -27198.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                         coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
Intercept                             -1.2281      0.079    

In [33]:
dogs["fixed"] = dogs["sex"].str.contains("Spayed|Neutered", na=False).astype(int)

m_age_fixed = smf.logit(
    "is_adopted ~ age_years * fixed",
    data=dogs
).fit()

print(m_age_fixed.summary())
print("\nOdds Ratios:\n", np.exp(m_age_fixed.params))


Optimization terminated successfully.
         Current function value: 0.465156
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:             is_adopted   No. Observations:                43252
Model:                          Logit   Df Residuals:                    43248
Method:                           MLE   Df Model:                            3
Date:                Sun, 30 Nov 2025   Pseudo R-squ.:                  0.2603
Time:                        21:17:56   Log-Likelihood:                -20119.
converged:                       True   LL-Null:                       -27198.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept          -2.9538      0.052    -57.328      0.000      -3.055      -2.853
age_years     

In [34]:
m_age_sex = smf.logit(
    "is_adopted ~ age_years * C(sex)",
    data=dogs
).fit()

print(m_age_sex.summary())
print("\nOdds Ratios:\n", np.exp(m_age_sex.params))


         Current function value: 0.464359
         Iterations: 35


/opt/conda/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


                           Logit Regression Results                           
Dep. Variable:             is_adopted   No. Observations:                43252
Model:                          Logit   Df Residuals:                    43238
Method:                           MLE   Df Model:                           13
Date:                Sun, 30 Nov 2025   Pseudo R-squ.:                  0.2615
Time:                        21:18:06   Log-Likelihood:                -20084.
converged:                      False   LL-Null:                       -27198.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                        coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
Intercept                            -2.8553      0.077    -37.233      0.000      -3.006      -2.705
C(sex)[T.Intact Female]              -2.2661      0.798     -2

NameError: name 'dogs_clean' is not defined